In [ ]:
from mp_api.client import MPRester
from dotenv import load_dotenv
import os
import pandas as pd
import numpy as np

load_dotenv()
API_KEY = os.getenv("MP_API_KEY")

/opt/homebrew/Caskroom/miniconda/base/envs/perovskite_optics/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# known perovskite-forming B-site elements
PEROVSKITE_B_SITES = {
    "Ti", "Zr", "Hf",   # group 4
    "V", "Nb", "Ta",    # group 5
    "Cr", "Mo", "W",    # group 6
    "Mn", "Fe", "Co",   # group 7-9
    "Ni", "Cu",         # group 10-11
    "Al", "Ga", "In",   # group 13
    "Sn", "Pb",         # group 14
    "Bi",               # group 15
}

with MPRester(API_KEY) as mpr:
    results = mpr.materials.summary.search(
        formula="**O3",
        fields=[
            "material_id",
            "formula_pretty",
            "band_gap",
            "formation_energy_per_atom",
            "energy_above_hull",
            "volume",
            "density",
            "elements",
            "nelements",
        ]
    )

df_oxide = pd.DataFrame([r.model_dump() for r in results])

# keep only 3-element compounds (A + B + O)
df_oxide = df_oxide[df_oxide["nelements"] == 3]

# keep only rows where at least one B-site element is present
def has_b_site(elements):
    return any(str(e) in PEROVSKITE_B_SITES for e in elements)

df_oxide = df_oxide[df_oxide["elements"].apply(has_b_site)]

print(f"Oxide perovskites after filtering: {len(df_oxide)}")
print(df_oxide["formula_pretty"].sample(min(15, len(df_oxide))).tolist())

Retrieving SummaryDoc documents: 100%|██████████| 2547/2547 [00:00<00:00, 3557.38it/s]

Oxide perovskites after filtering: 1455
['TmVO3', 'NbTlO3', 'ZnCrO3', 'ErCuO3', 'PrAlO3', 'BaTiO3', 'NaVO3', 'LaInO3', 'NiCO3', 'NaBiO3', 'KBiO3', 'MnBO3', 'LiAlO3', 'ScBiO3', 'AlFeO3']


In [9]:
with MPRester(API_KEY) as mpr:
    results_halide = mpr.materials.summary.search(
        formula="**X3",          # X = halide placeholder
        chemsys=["Cs-Pb-I", "Cs-Pb-Br", "Cs-Pb-Cl", "Cs-Sn-I", "Cs-Sn-Br", "Cs-Sn-Cl",
                 "Rb-Pb-I", "Rb-Pb-Br", "K-Pb-I"],  # halide perovskites
        fields=[
            "material_id",
            "formula_pretty",
            "band_gap",
            "formation_energy_per_atom",
            "energy_above_hull",
            "volume",
            "density",
            "elements",
            "nelements",
        ]
    )
df_halide = pd.DataFrame([r.model_dump() for r in results_halide])
print(f"Halide perovskites pulled: {len(df_halide)}")
df_halide.head()

Retrieving SummaryDoc documents: 100%|██████████| 24/24 [00:00<00:00, 296068.52it/s]

Halide perovskites pulled: 24


,elements,nelements,formula_pretty,volume,density,material_id,formation_energy_per_atom,energy_above_hull,band_gap,fields_not_requested
0,"[Br, Cs, Pb]",3,CsPbBr3,406.141335,4.741254,mp-cfsgm,-1.626886,0.007484,1.8178,"[builder_meta, nsites, composition, compositio..."
1,"[Br, Cs, Pb]",3,CsPbBr3,407.629403,4.723945,mp-cltwv,-1.625293,0.009077,1.9550,"[builder_meta, nsites, composition, compositio..."
2,"[Br, Cs, Pb]",3,CsPbBr3,813.530535,4.733981,mp-bghrx,-1.629785,0.004584,2.0121,"[builder_meta, nsites, composition, compositio..."
3,"[Br, Cs, Pb]",3,CsPbBr3,769.207176,5.006763,mp-bghtx,-1.634370,0.000000,2.8808,"[builder_meta, nsites, composition, compositio..."
4,"[Br, Cs, Pb]",3,CsPbBr3,210.499491,4.573928,mp-bidsj,-1.609297,0.025073,1.7819,"[builder_meta, nsites, composition, compositio..."


In [10]:
df_oxide["family"] = "oxide"
df_halide["family"] = "halide"
df = pd.concat([df_oxide, df_halide], ignore_index=True)

df = df[df["energy_above_hull"] < 0.1]  # filter for stability
df = df[df["band_gap"] > 0]  # filter for non-metals
df = df[df["band_gap"].notna()]  # filter for entries with band gap data

df = df.reset_index(drop=True)
print(f"Final dataset size after filtering: {len(df)}")
print(df[["formula_pretty", "band_gap", "family"]].head(10))

Final dataset size after filtering: 603
  formula_pretty  band_gap family
0           VHO3    0.0094  oxide
1         CoNiO3    0.7645  oxide
2         LaCoO3    0.6739  oxide
3          VCoO3    1.1514  oxide
4          NbHO3    2.6221  oxide
5         TiNiO3    1.4963  oxide
6           VHO3    2.1077  oxide
7          NbHO3    2.5310  oxide
8          NbHO3    0.0005  oxide
9           VHO3    0.6363  oxide


In [11]:
print(f"Total rows: {len(df)}")
print(f"\nFamily breakdown:")
print(df["family"].value_counts())
print(f"\nBand gap range:")
print(df["band_gap"].describe())
print(f"\nSample formulas:")
print(df["formula_pretty"].sample(15).tolist())

Total rows: 603

Family breakdown:
family
oxide     579
halide     24
Name: count, dtype: int64

Band gap range:
count    603.000000
mean       2.036453
std        1.222286
min        0.000200
25%        1.080150
50%        1.996200
75%        2.901850
max        6.091400
Name: band_gap, dtype: float64

Sample formulas:
['LiVO3', 'YCoO3', 'CaTiO3', 'GePbO3', 'TbCoO3', 'HfFeO3', 'MnBO3', 'VHO3', 'NaVO3', 'NaNbO3', 'FeSiO3', 'ZnSnO3', 'ScAlO3', 'NbHO3', 'TiZnO3']


In [12]:
from pymatgen.core import Element

def get_features(row):
    try:
        elements = [str(e) for e in row["elements"]]
        # remove oxygen for oxides to isolate A and B sites
        cations = [e for e in elements if e != "O" and e not in ["I","Br","Cl","F"]]
        
        if len(cations) < 2:
            return None
            
        # get pymatgen Element objects
        els = [Element(e) for e in cations]
        
        radii = [e.average_ionic_radius for e in els]
        electroneg = [e.X for e in els]
        
        return {
            "material_id": row["material_id"],
            "formula": row["formula_pretty"],
            "family": row["family"],
            "band_gap": row["band_gap"],
            "formation_energy": row["formation_energy_per_atom"],
            "volume": row["volume"],
            "density": row["density"],
            "mean_ionic_radius": np.mean([r for r in radii if r is not None]),
            "radius_variance": np.var([r for r in radii if r is not None]),
            "mean_electronegativity": np.mean(electroneg),
            "electronegativity_diff": max(electroneg) - min(electroneg),
            "n_elements": row["nelements"],
        }
    except Exception:
        return None

feature_rows = [get_features(row) for _, row in df.iterrows()]
df_features = pd.DataFrame([r for r in feature_rows if r is not None])

df_features = df_features.dropna()
df_features = df_features.reset_index(drop=True)

print(f"Feature matrix shape: {df_features.shape}")
print(df_features.head())

Feature matrix shape: (603, 12)
  material_id formula family  band_gap  formation_energy      volume  \
0    mp-cpbyc    VHO3  oxide    0.0094         -1.922323  262.154142   
1    mp-cukac  CoNiO3  oxide    0.7645         -1.113034   96.029107   
2    mp-cukub  LaCoO3  oxide    0.6739         -2.569496  237.877427   
3    mp-curdd   VCoO3  oxide    1.1514         -1.922899  103.061900   
4     mp-byfc   NbHO3  oxide    2.6221         -2.474307  235.873732   

    density  mean_ionic_radius  radius_variance  mean_electronegativity  \
0  2.532357           0.388750         0.151127                   1.915   
1  5.727981           0.754167         0.000201                   1.895   
2  6.864404           0.970167         0.040737                   1.490   
3  5.087314           0.772917         0.000021                   1.755   
4  3.996228           0.410000         0.168100                   1.900   

   electronegativity_diff  n_elements  
0                    0.57           3  
1   